In [2]:
import pandas as pd
import numpy as np

In [ ]:
from google.colab import files
uploaded = files.upload()

Saving messi barca.csv to messi barca.csv


In [ ]:
df = pd.read_csv('messi barca.csv', encoding='latin1')
print(df.head())

      Season  Age       Competition  Matches Played  Starts  Minutes played  \
0  2004-2005   17  Champions League               1       1              90   
1  2004-2005   17           La Liga               7       0              70   
2  2005-2006   18  Champions League               6       4             322   
3  2005-2006   18           La Liga              17      11             911   
4  2006-2007   19  Champions League               5       4             385   

   Goals scored  Assists  PK  PKatt  CrdY  CrdR  Goal/90  Ast/90  G+A/90  \
0             0        0   0      0     0     0     0.00    0.00    0.00   
1             1        0   0      0     0     0     1.29    0.00    1.29   
2             1        1   0      0     0     0     0.28    0.28    0.56   
3             6        3   0      0     2     0     0.59    0.30    0.89   
4             1        0   0      0     1     0     0.23    0.00    0.23   

   G-PK/90  G+A-PK/90  
0     0.00       0.00  
1     1.29       1.2

In [ ]:
print("--- 1. Missing values per column ---")
print(df.isnull().sum())

--- 1. Missing values per column ---
Season            0
Age               0
Competition       0
Matches Played    0
Starts            0
Minutes played    0
Goals scored      0
Assists           0
PK                0
PKatt             0
CrdY              0
CrdR              0
Goal/90           0
Ast/90            0
G+A/90            0
G-PK/90           0
G+A-PK/90         0
dtype: int64


In [ ]:
print("\n--- 2. Checking for duplicates ---")
print("Number of full duplicates before artificial injection:", df.duplicated().sum())


--- 2. Checking for duplicates ---
Number of full duplicates before artificial injection: 0


In [ ]:
np.random.seed(42)
nan_indices = np.random.choice(df.index, 3, replace=False)
df.loc[nan_indices, 'Goals scored'] = np.nan

In [ ]:
df = pd.concat([df, df.head(2)], ignore_index=True)
print("Number of duplicates after artificial injection:", df.duplicated().sum())

Number of duplicates after artificial injection: 2


In [ ]:
df = df.drop_duplicates()
df['Goals scored'] = df['Goals scored'].fillna(df['Goals scored'].median())

In [ ]:
df['Competition'] = df['Competition'].str.strip().str.lower()

In [ ]:
df['Start_Year'] = df['Season'].str.split('-').str[0].astype(int)

In [ ]:
df['G+A_per_match'] = np.where(
    df['Matches Played'] > 0,
    (df['Goals scored'] + df['Assists']) / df['Matches Played'],
    0
)

In [ ]:
df_encoded = pd.get_dummies(df, columns=['Competition'])
print("\n--- 4. First rows after One-hot encoding for competitions ---")
display(df_encoded[['Season', 'Goals scored', 'Competition_la liga']].head())


--- 4. First rows after One-hot encoding for competitions ---


,Season,Goals scored,Competition_la liga
0,2004-2005,0.0,False
1,2004-2005,1.0,True
2,2005-2006,1.0,False
3,2005-2006,6.0,True
4,2006-2007,1.0,False


In [ ]:
min_g = df['Goals scored'].min()
max_g = df['Goals scored'].max()
df['goals_norm'] = (df['Goals scored'] - min_g) / (max_g - min_g)

In [ ]:
mean_g = df['Goals scored'].mean()
std_g = df['Goals scored'].std()
df['goals_std'] = (df['Goals scored'] - mean_g) / std_g

In [ ]:
print("\n--- 5. Verification of normalization and standardization ---")
print(f"Minimum goals_norm: {df['goals_norm'].min():.1f}")
print(f"Maximum goals_norm: {df['goals_norm'].max():.1f}")
print(f"Mean goals_std (approx. 0): {df['goals_std'].mean():.2e}")
print(f"Standard deviation goals_std: {df['goals_std'].std():.1f}")


--- 5. Verification of normalization and standardization ---
Minimum goals_norm: 0.0
Maximum goals_norm: 1.0
Mean goals_std (approx. 0): 8.58e-17
Standard deviation goals_std: 1.0


In [ ]:
df['is_top_scoring_season'] = np.where(df['Goals scored'] >= 30, 1, 0)

In [ ]:
final_columns = ['Season', 'Age', 'Competition', 'Matches Played', 'Goals scored',
                 'Assists', 'goals_norm', 'goals_std', 'is_top_scoring_season']
final_df = df[final_columns]

final_df.to_csv('cleaned_messi_barca_final.csv', index=False)
print("\nThe cleaned and processed dataset has been successfully saved to 'cleaned_messi_barca_final.csv' without indices!")


The cleaned and processed dataset has been successfully saved to 'cleaned_messi_barca_final.csv' without indices!


Task 1: Create a custom DataFrame with defects and clean it

In [3]:
data = {
    'id': [1, 2, 3, 4, 2, 5, 6],
    'category': ['A', 'B', 'A', None, 'B', 'A', 'A'],
    'value': [10, 15, 12, 11, 15, 250, 14],
    'score': [85.5, None, 88.0, 90.5, None, 82.0, 89.0],
    'status': ['active', 'pending', 'active', 'inactive', 'pending', 'active', 'active']
}
df_custom = pd.DataFrame(data)

In [4]:
print("--- Original DataFrame with defects ---")
display(df_custom)

--- Original DataFrame with defects ---


,id,category,value,score,status
0,1,A,10,85.5,active
1,2,B,15,NaN,pending
2,3,A,12,88.0,active
3,4,None,11,90.5,inactive
4,2,B,15,NaN,pending
5,5,A,250,82.0,active
6,6,A,14,89.0,active


In [5]:
df_custom = df_custom.drop_duplicates()

In [6]:
for col in df_custom.select_dtypes(include=[np.number]).columns:
    median_val = df_custom[col].median()
    df_custom[col] = df_custom[col].fillna(median_val)

for col in df_custom.select_dtypes(include=['object', 'category']).columns:
    if not df_custom[col].mode().empty:
        mode_val = df_custom[col].mode()[0]
        df_custom[col] = df_custom[col].fillna(mode_val)

In [7]:
for col in df_custom.select_dtypes(include=[np.number]).columns:
    Q1 = df_custom[col].quantile(0.25)
    Q3 = df_custom[col].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    df_custom = df_custom[(df_custom[col] >= lower_bound) & (df_custom[col] <= upper_bound)]

print("\n--- Cleaned DataFrame ---")
display(df_custom)


--- Cleaned DataFrame ---


,id,category,value,score,status
1,2,B,15,88.0,pending
2,3,A,12,88.0,active
3,4,A,11,90.5,inactive
6,6,A,14,89.0,active


Additional Task: Universal clean_dataset(df) function

In [9]:
def clean_dataset(df):
    df_clean = df.copy()
    df_clean = df_clean.drop_duplicates()
    for col in df_clean.select_dtypes(include=[np.number]).columns:
        df_clean[col] = df_clean[col].fillna(df_clean[col].median())
    for col in df_clean.select_dtypes(include=['object', 'category']).columns:
        if not df_clean[col].mode().empty:
            df_clean[col] = df_clean[col].fillna(df_clean[col].mode()[0])
    for col in df_clean.columns:
        if 'date' in col.lower():
            df_clean[col] = pd.to_datetime(df_clean[col], errors='coerce')
    for col in df_clean.select_dtypes(include=[np.number]).columns:
        Q1 = df_clean[col].quantile(0.25)
        Q3 = df_clean[col].quantile(0.75)
        IQR = Q3 - Q1
        lower_bound = Q1 - 1.5 * IQR
        upper_bound = Q3 + 1.5 * IQR
        df_clean = df_clean[(df_clean[col] >= lower_bound) & (df_clean[col] <= upper_bound)]

    return df_clean

In [10]:
synthetic_dirty_df = pd.DataFrame({
    'date': ['2024-03-01', '2024-03-02', None, '2024-03-02', '2024-03-05'],
    'category': ['Latte', 'Espresso', None, 'Espresso', 'Cappuccino'],
    'price': [35.0, 25.0, 30.0, 500.0, 28.0], # 500.0 is an extreme outlier
    'rating': [5, 4, None, 4, 5]
})

In [11]:
print("--- Original Synthetic Dirty Dataset ---")
display(synthetic_dirty_df)

--- Original Synthetic Dirty Dataset ---


,date,category,price,rating
0,2024-03-01,Latte,35.0,5.0
1,2024-03-02,Espresso,25.0,4.0
2,None,None,30.0,NaN
3,2024-03-02,Espresso,500.0,4.0
4,2024-03-05,Cappuccino,28.0,5.0


In [12]:
cleaned_result_df = clean_dataset(synthetic_dirty_df)

In [13]:
print("\n--- Result After clean_dataset() ---")
display(cleaned_result_df)


--- Result After clean_dataset() ---


,date,category,price,rating
0,2024-03-01,Latte,35.0,5.0
1,2024-03-02,Espresso,25.0,4.0
2,2024-03-02,Espresso,30.0,4.5
4,2024-03-05,Cappuccino,28.0,5.0


In [14]:
cleaned_result_df.to_csv('cleaned_tested_dataset.csv', index=False)
print("\nThe cleaned dataset has been successfully saved to 'cleaned_tested_dataset.csv'!")


The cleaned dataset has been successfully saved to 'cleaned_tested_dataset.csv'!
